# PDEBench-Lang: Dataset Generation
**Raghav Anand | CSCI 5541 NLP S26**

Generates `dataset.jsonl` — 10,000 instances across 5 PDE families (2000 each).

Each instance has 4 dialect representations: LaTeX, Prefix, Postfix, Natural Language.

**Fix applied**: Laplace now uses `k*(u_xx + u_yy) = 0` with `k ~ Uniform(0.1, 3.0)` so every instance has a unique symbolic expression across all dialects.

In [ ]:
# Cell 1: Install
import sys
!{sys.executable} -m pip install -q sympy

In [ ]:
# Cell 2: Templates
# Each family defines:
#   coefficients — names of coefficients to randomly sample
#   range        — uniform sampling range for those coefficients
#   nl_templates — natural language phrasings (one picked at random per instance)
#   reasoning_templates — reasoning chain phrasings (one picked at random)
#   operators    — fixed operator set label for that family

PDE_TEMPLATES = {
    'Heat': {
        'coefficients': ['alpha'], 'range': (0.1, 3.0),
        'nl_templates': [
            'The time derivative of u equals {alpha} times the second spatial derivative of u.',
            'u changes in time at a rate proportional to {alpha} times its spatial curvature.',
            'The rate of change of u with respect to time is {alpha} multiplied by the second derivative of u with respect to x.',
            'Diffusion governs u: its temporal evolution equals {alpha} times its second-order spatial variation.',
            'u_t equals {alpha} times u_xx, describing how u spreads over time due to diffusion.',
            'The temporal rate of change of u is driven by {alpha} times the curvature of u in space.',
        ],
        'reasoning_templates': [
            'Contains a first-order time derivative and second-order spatial derivative, indicating diffusion with coefficient {alpha}.',
            'Structure: u_t = {alpha} * u_xx. First-order in time, second-order in space — heat equation with diffusivity {alpha}.',
            'A single time derivative balanced against a second spatial derivative scaled by {alpha} identifies this as diffusive transport.',
            'First-order temporal derivative and second-order spatial derivative with diffusion coefficient {alpha} — classic heat equation.',
        ],
        'operators': ['exp', 'polynomial'],
    },
    'Wave': {
        'coefficients': ['c'], 'range': (0.1, 3.0),
        'nl_templates': [
            'The second time derivative of u equals {c} squared times the second spatial derivative of u.',
            'u oscillates such that its second temporal derivative equals {c}^2 times its second spatial derivative.',
            'Wave propagation with speed {c}: the second-order time change equals {c}^2 times the second-order spatial change.',
            'u_tt equals {c}^2 times u_xx, governing oscillation at propagation speed {c}.',
            'The curvature of u in time is proportional to {c}^2 times its curvature in space.',
            'The acceleration of u in time equals {c} squared times its spatial curvature.',
        ],
        'reasoning_templates': [
            'Contains a second-order time derivative and second-order spatial derivative, indicating wave propagation with speed {c}.',
            'The equation is second-order in both time and space: u_tt = {c}^2 * u_xx — wave equation with speed {c}.',
            'Second-order temporal and spatial derivatives both present, with speed coefficient {c}, characteristic of the wave equation.',
            'Two time derivatives on the left, two spatial derivatives scaled by {c}^2 on the right — defining pattern of a wave equation.',
        ],
        'operators': ['sin', 'cos', 'polynomial'],
    },
    'Burgers': {
        'coefficients': ['nu'], 'range': (0.05, 1.5),
        'nl_templates': [
            'The time derivative of u plus u times its spatial derivative equals {nu} times the second spatial derivative of u.',
            'Nonlinear advection and diffusion: u_t plus u times u_x balances against {nu} times u_xx.',
            'u_t plus the nonlinear term u*u_x equals {nu} times u_xx, capturing both nonlinear transport and diffusion.',
            'The sum of u_t and the product u times u_x equals {nu} times the second spatial derivative of u.',
            'u evolves via self-advection: its time rate plus u times its gradient equals {nu} times its second spatial derivative.',
            'The temporal change of u plus the nonlinear convection term u*u_x equals viscous diffusion scaled by {nu}.',
        ],
        'reasoning_templates': [
            'Contains nonlinear convection term u*u_x and diffusion term with coefficient {nu}.',
            'Nonlinear first-order spatial term u*u_x combined with second-order diffusion {nu}*u_xx identifies this as Burgers equation.',
            'The u*u_x term introduces nonlinearity; the {nu}*u_xx term provides diffusion — Burgers equation with viscosity {nu}.',
            'Nonlinear advection (u*u_x) plus linear diffusion ({nu}*u_xx) — uniquely characteristic of Burgers equation.',
        ],
        'operators': ['tanh', 'polynomial'],
    },
    # FIX: Laplace originally had no coefficients so every instance was identical.
    # We add k ~ Uniform(0.1, 3.0) and use k*(u_xx + u_yy) = 0.
    # This is mathematically equivalent to u_xx + u_yy = 0 for any k != 0
    # but gives every instance a unique LaTeX/prefix/postfix/natural string.
    'Laplace': {
        'coefficients': ['k'], 'range': (0.1, 3.0),
        'nl_templates': [
            'The sum of the second spatial derivatives in x and y, scaled by {k}, equals zero.',
            '{k} times the second derivative of u in x plus {k} times the second derivative of u in y equals zero.',
            'u has zero Laplacian scaled by {k}: its spatial curvature in x and y sum to zero.',
            'Steady-state equilibrium with scaling {k}: the second spatial derivatives in x and y cancel to zero.',
            '{k} times u_xx plus {k} times u_yy equals zero, indicating steady-state with no temporal evolution.',
            'The curvature of u in x plus the curvature in y, scaled by {k}, equals zero.',
            'With diffusion constant {k}, the Laplacian of u vanishes: {k}*(u_xx + u_yy) = 0.',
            'u satisfies a scaled Laplace equation: {k} times its second x-derivative plus {k} times its second y-derivative is zero.',
            'No time dependence; {k} times spatial curvature in x plus {k} times spatial curvature in y equals zero.',
            'The steady-state condition {k}*(u_xx + u_yy) = 0 holds, making u harmonic up to scale factor {k}.',
        ],
        'reasoning_templates': [
            'Contains second-order spatial derivatives in x and y scaled by {k} — mathematically equivalent to Laplace equation.',
            'No time derivative present; only {k}*(u_xx + u_yy) = 0, indicating steady-state equilibrium.',
            'The absence of any time derivative and presence of {k}*(u_xx + u_yy) = 0 identifies this as Laplace equation.',
            'Scale factor {k} multiplies both spatial second-order derivatives; their sum is zero — harmonic (Laplace) structure.',
        ],
        'operators': ['sin', 'cos', 'exp', 'polynomial'],
    },
    'Advection': {
        'coefficients': ['c'], 'range': (0.1, 3.0),
        'nl_templates': [
            'The time derivative of u plus {c} times its spatial derivative equals zero.',
            'u is transported at velocity {c}: its temporal change plus {c} times its spatial change equals zero.',
            'u propagates without diffusion: u_t plus {c} times u_x equals zero.',
            'The first-order time derivative plus {c} times the first-order spatial derivative equals zero, describing advection at speed {c}.',
            'u moves at constant speed {c}: the sum of u_t and {c} times u_x is zero.',
            'Pure transport: the rate of change of u in time plus {c} times the rate of change in space is zero.',
        ],
        'reasoning_templates': [
            'Contains first-order time and spatial derivatives representing pure transport with velocity {c}.',
            'Only first-order derivatives: u_t and {c}*u_x, no u_xx — pure transport equation with velocity {c}.',
            'Absence of second-order derivatives and presence of u_t + {c}*u_x = 0 identifies this as advection equation.',
            'No second-order spatial derivative; only u_t and {c}*u_x appear — advection at speed {c}.',
        ],
        'operators': ['exp', 'polynomial'],
    },
}

print('Templates loaded for families:', list(PDE_TEMPLATES.keys()))

In [ ]:
# Cell 3: Coefficient sampler
# For each family, randomly samples coefficient values from Uniform(low, high)
# Laplace now samples k instead of returning empty dict

import random

def sample_coefficients(family):
    tmpl = PDE_TEMPLATES[family]
    lo, hi = tmpl['range']
    return {name: round(random.uniform(lo, hi), 2) for name in tmpl['coefficients']}

# Quick check
for fam in PDE_TEMPLATES:
    print(f'{fam}: {sample_coefficients(fam)}')

In [ ]:
# Cell 4: Equation builder
# Uses SymPy to build the actual symbolic equation object from sampled coefficients.
# Laplace fix: k*u_xx + k*u_yy = 0  (was: u_xx + u_yy = 0, same every time)

from sympy import symbols, Function, Eq, Derivative, latex as sym_latex

x, y, t = symbols('x y t')
u = Function('u')

def build_equation(family, coeffs):
    if family == 'Heat':
        return Eq(Derivative(u(t,x), t), coeffs['alpha'] * Derivative(u(t,x), x, 2))
    if family == 'Wave':
        c = coeffs['c']
        return Eq(Derivative(u(t,x), t, 2), c**2 * Derivative(u(t,x), x, 2))
    if family == 'Burgers':
        return Eq(Derivative(u(t,x), t) + u(t,x)*Derivative(u(t,x), x),
                  coeffs['nu'] * Derivative(u(t,x), x, 2))
    if family == 'Laplace':
        k = coeffs['k']
        # FIX: k*(u_xx + u_yy) = 0 instead of u_xx + u_yy = 0
        return Eq(k*Derivative(u(x,y), x, 2) + k*Derivative(u(x,y), y, 2), 0)
    if family == 'Advection':
        return Eq(Derivative(u(t,x), t) + coeffs['c']*Derivative(u(t,x), x), 0)

# Preview equations
for fam in PDE_TEMPLATES:
    coeffs = sample_coefficients(fam)
    eq = build_equation(fam, coeffs)
    print(f'{fam} {coeffs}: {sym_latex(eq)}')

In [ ]:
# Cell 5: Dialect converters
# Converts a SymPy equation object into Prefix and Postfix notation
# by recursively walking the expression tree.

def to_prefix(expr):
    from sympy import Eq, Add, Mul, Derivative, Pow
    if expr.is_Number or expr.is_Symbol: return str(expr)
    if expr.func == Eq:   return f'=({to_prefix(expr.lhs)}, {to_prefix(expr.rhs)})'
    if expr.func == Add:
        args = [to_prefix(a) for a in expr.args]
        r = args[0]
        for a in args[1:]: r = f'+({r}, {a})'
        return r
    if expr.func == Mul:
        args = [to_prefix(a) for a in expr.args]
        r = args[0]
        for a in args[1:]: r = f'*({r}, {a})'
        return r
    if expr.func == Pow:
        return f'^({to_prefix(expr.args[0])}, {to_prefix(expr.args[1])})'
    if expr.func == Derivative:
        r = to_prefix(expr.args[0])
        var = str(expr.args[1][0])
        order = int(expr.args[1][1]) if len(expr.args[1]) > 1 else 1
        for _ in range(order): r = f'd({r}, {var})'
        return r
    return str(expr)

def to_postfix(expr):
    from sympy import Eq, Add, Mul, Derivative, Pow
    if expr.is_Number or expr.is_Symbol: return str(expr)
    if expr.func == Eq:   return f'{to_postfix(expr.lhs)} {to_postfix(expr.rhs)} ='
    if expr.func == Add:
        args = [to_postfix(a) for a in expr.args]
        r = args[0]
        for a in args[1:]: r = f'{r} {a} +'
        return r
    if expr.func == Mul:
        args = [to_postfix(a) for a in expr.args]
        r = args[0]
        for a in args[1:]: r = f'{r} {a} *'
        return r
    if expr.func == Pow:
        return f'{to_postfix(expr.args[0])} {to_postfix(expr.args[1])} ^'
    if expr.func == Derivative:
        r = to_postfix(expr.args[0])
        var = str(expr.args[1][0])
        order = int(expr.args[1][1]) if len(expr.args[1]) > 1 else 1
        for _ in range(order): r = f'{r} {var} d'
        return r
    return str(expr)

# Preview all 4 dialects for one Laplace instance
coeffs = {'k': 1.5}
eq = build_equation('Laplace', coeffs)
print('LaTeX  :', sym_latex(eq))
print('Prefix :', to_prefix(eq))
print('Postfix:', to_postfix(eq))

In [ ]:
# Cell 6: Instance generator
# Combines all the above into one instance with all 4 dialects and labels.

def generate_instance(family):
    tmpl  = PDE_TEMPLATES[family]
    coeffs = sample_coefficients(family)
    eq     = build_equation(family, coeffs)
    return {
        'family'      : family,
        'coefficients': coeffs,
        'dialects': {
            'latex'  : sym_latex(eq),
            'prefix' : to_prefix(eq),
            'postfix': to_postfix(eq),
            'natural': random.choice(tmpl['nl_templates']).format(**coeffs),
        },
        'labels': {
            'behavioral': family,
            'operators' : tmpl['operators'],
            'reasoning' : random.choice(tmpl['reasoning_templates']).format(**coeffs),
        },
    }

# Preview one instance per family
import json
random.seed(42)
for fam in PDE_TEMPLATES:
    inst = generate_instance(fam)
    print(f'\n=== {fam} ===')
    print(json.dumps(inst, indent=2))

In [ ]:
# Cell 7: Generate full dataset and save
# 2000 instances per family x 5 families = 10,000 total
# Shuffled and saved to /content/dataset.jsonl

random.seed(42)
FAMILIES = ['Heat', 'Wave', 'Burgers', 'Laplace', 'Advection']
INSTANCES_PER_FAMILY = 2000
OUTPUT_PATH = '/content/dataset.jsonl'

all_instances = []
for family in FAMILIES:
    for _ in range(INSTANCES_PER_FAMILY):
        all_instances.append(generate_instance(family))

random.shuffle(all_instances)

with open(OUTPUT_PATH, 'w') as f:
    for inst in all_instances:
        f.write(json.dumps(inst) + '\n')

print(f'Saved {len(all_instances)} instances to {OUTPUT_PATH}')

In [ ]:
# Cell 8: Verify diversity
# Checks how many unique strings exist per dialect per family.
# Before fix: Laplace had 1 unique latex, 6 unique natural.
# After fix:  Laplace has ~291 unique latex, ~1400+ unique natural.

from collections import Counter

with open(OUTPUT_PATH) as f:
    data = [json.loads(l) for l in f]

print(f'Total instances: {len(data)}')
print(f'Family distribution: {Counter(d["family"] for d in data)}')
print()
print(f'{"Family":<12} {"latex":>8} {"prefix":>8} {"postfix":>8} {"natural":>8}')
print('-' * 50)
for family in FAMILIES:
    subset = [d for d in data if d['family'] == family]
    u_latex   = len(set(d['dialects']['latex']   for d in subset))
    u_prefix  = len(set(d['dialects']['prefix']  for d in subset))
    u_postfix = len(set(d['dialects']['postfix'] for d in subset))
    u_natural = len(set(d['dialects']['natural'] for d in subset))
    print(f'{family:<12} {u_latex:>8} {u_prefix:>8} {u_postfix:>8} {u_natural:>8}')